In [1]:
import itertools
import json
import sys
import gc
from pathlib import Path
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [2]:
EPOCHS = 1000
BATCH_SIZE = 64
ACT_FUNC = tf.nn.relu
MOMENTUM = 0.5

In [3]:
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [[1024, 1024, 1024], [1024, 1024, 512], [512, 512], [512, 512, 512], [2048, 1024, 512], ]
lr_options = [1e-4, 2e-4, 3e-4]
dropout_options = [(0.2, 0.3), (0.2, 0.5)]
hyperparameter_grid = list(itertools.product(norm_options, hidden_options, lr_options, dropout_options))
print(f"Total no. of hyperparameter combinations: {len(hyperparameter_grid)}")

Total no. of hyperparameter combinations: 90


In [4]:
checkpoint_file = ROOT_DIR / "hyperparam_checkpoint.json"
best_val_loss = np.inf
best_params = None
best_epoch = None
start_idx = 0
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        checkpoint = json.load(f)
    best_val_loss = checkpoint.get("best_val_loss", np.inf)
    best_params = checkpoint.get("best_params", None)
    best_epoch = checkpoint.get("best_epoch", None)
    start_idx = checkpoint.get("last_completed_idx", -1) + 1
    records = checkpoint.get("records", [])
    print(f"Resuming from index {start_idx}, best_val_loss so far: {best_val_loss}")
else:
    records = []

In [5]:
data_cache = {}
for norm in norm_options:
    train_features, val_features, _, _, train_targets, val_targets, _, _ = load(norm=norm)
    data_cache[norm] = (train_features, val_features, train_targets, val_targets)
    print(f"\n{norm}")
    print("Train features shape:", train_features.shape)
    print("Val features shape:", val_features.shape)
    print("Train targets shape:", train_targets.shape)
    print("Val targets shape:", val_targets.shape)
    print("NaN in train_features:", np.isnan(train_features).any())
    print("Inf in train_features:", np.isinf(train_features).any())
    print("NaN in train_targets:", np.isnan(train_targets).any())
    print("Inf in train_targets:", np.isinf(train_targets).any())
    print("\nFirst 5 rows of train_features:\n", train_features[:5])
    print("First 5 elements of train_targets:\n", train_targets[:5])



norm
Train features shape: (13884, 7060)
Val features shape: (4614, 7060)
Train targets shape: (13884, 1)
Val targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in train_targets: False

First 5 rows of train_features:
 [[-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]
 [-0.38594985 -0.54272044 -0.2307692  ... -0.561097   -0.68233347
   0.07810206]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.4991565   1.9423046
  -0.38811162]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]
 [-0.38594985 -0.54272044 -0.2307692  ...  0.          0.
   0.        ]]
First 5 elements of train_targets:
 [[ 7.69353  ]
 [ 7.7780533]
 [-1.1985054]
 [ 2.5956845]
 [-5.1399713]]

tanh
Train features shape: (13884, 7060)
Val features shape: (4614, 7060)
Train targets shape: (13884, 1)
Val targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in

In [6]:
def moving_average(x, n):
    return np.convolve(x, np.ones(n) / n, mode='valid')

In [7]:
for idx, (norm_type, hidden_layers, lr, (input_dropout, hidden_dropout)) in enumerate(hyperparameter_grid):
    if idx < start_idx:
        continuessa

    train_features, val_features, train_targets, val_targets = data_cache[norm_type]

    K.clear_session()
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units, input_shape=(train_features.shape[1],), activation=ACT_FUNC, kernel_initializer='he_normal'))
            if input_dropout > 0:
                model.add(Dropout(float(input_dropout)))
        else:
            model.add(Dense(
                units, activation=ACT_FUNC, kernel_initializer='he_normal'))
            if hidden_dropout > 0:
                model.add(Dropout(float(hidden_dropout)))

    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))
    model.compile(
        loss='mean_squared_error',
        optimizer=SGD(learning_rate=float(lr), momentum=MOMENTUM)
    )
    model.summary()

    history = model.fit(
        train_features, train_targets,
        validation_data=(val_features, val_targets),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()]
    )

    val_losses = np.array(history.history['val_loss'])
    local_best_epoch = int(np.argmin(val_losses))
    local_best_loss = float(val_losses[local_best_epoch])

    if local_best_loss < best_val_loss:
        best_val_loss = local_best_loss
        best_epoch = local_best_epoch + 1
        best_params = {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
            "epochs": best_epoch
        }

    records.append({
        "local_params": {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
        },
        "local_best_epoch": local_best_epoch,
        "local_best_loss": None if np.isnan(local_best_loss) else local_best_loss
    })

    checkpoint_data = {
        "last_completed_idx": idx,
        "best_val_loss": float(best_val_loss),
        "best_params": best_params,
        "best_epoch": best_epoch,
        "records": records
    }
    with open(checkpoint_file, "w") as f:
        json.dump(checkpoint_data, f, indent=2)

    del model
    del history
    K.clear_session()
    gc.collect()
    tf.compat.v1.reset_default_graph()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1024)              7230464   
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 1024)              1049600   
                                                                 
 dropout_1 (Dropout)         (None, 1024)              0         
                                                                 
 dense_2 (Dense)             (None, 1024)              1049600   
                                                                 
 dropout_2 (Dropout)         (None, 1024)              0         
                                                                 
 dense_3 (Dense)             (None, 1)                 1

I0000 00:00:1771572646.504958     318 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771572646.505007     318 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771572646.505016     318 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771572646.655390     318 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771572646.655429     318 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-02-20

Epoch 1/1000


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

 17/217 [=>............................] - ETA: 0s - loss: 612.6682   

...

217/217 [==============================] - 1s 3ms/step - loss: 487.0556 - val_loss: 375.7739
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 435.2512 - val_loss: 360.3286
Epoch 3/1000
217/217 [==============================] - 1s 3ms/step - loss: 408.1507 - val_loss: 361.0342
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 390.4539 - val_loss: 351.0335
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 372.6386 - val_loss: 385.3506
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 369.1433 - val_loss: 376.4348
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 362.7780 - val_loss: 346.1933
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 354.6268 - val_loss: 344.3068
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 359.3992 - val_loss: 348.4603
Epoch 10/1000
217/217 [==============================] - 1s

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

217/217 [==============================] - 1s 4ms/step - loss: 467.2866 - val_loss: 368.1724
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 414.3932 - val_loss: 348.8543
Epoch 3/1000
217/217 [==============================] - 1s 3ms/step - loss: 389.3431 - val_loss: 354.8466
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 372.2451 - val_loss: 354.4330
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 358.5551 - val_loss: 367.5453
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 359.2387 - val_loss: 354.4821
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 357.8572 - val_loss: 355.4112
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 358.3258 - val_loss: 358.2304
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 356.8054 - val_loss: 352.7680
Epoch 10/1000
217/217 [==============================] - 1s

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

217/217 [==============================] - 1s 3ms/step - loss: 502.3080 - val_loss: 396.8994
Epoch 2/1000
217/217 [==============================] - 1s 3ms/step - loss: 455.4768 - val_loss: 360.0991
Epoch 3/1000
217/217 [==============================] - 3s 16ms/step - loss: 429.2838 - val_loss: 363.5990
Epoch 4/1000
217/217 [==============================] - 1s 3ms/step - loss: 408.2200 - val_loss: 358.5184
Epoch 5/1000
217/217 [==============================] - 1s 3ms/step - loss: 403.4828 - val_loss: 370.7158
Epoch 6/1000
217/217 [==============================] - 1s 3ms/step - loss: 399.2654 - val_loss: 376.4917
Epoch 7/1000
217/217 [==============================] - 1s 3ms/step - loss: 401.8603 - val_loss: 369.0252
Epoch 8/1000
217/217 [==============================] - 1s 3ms/step - loss: 395.3758 - val_loss: 412.6881
Epoch 9/1000
217/217 [==============================] - 1s 3ms/step - loss: 405.0760 - val_loss: 344.0343
Epoch 10/1000
217/217 [==============================] - 1

In [8]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"best_val_loss: {best_val_loss}\n")

In [2]:
checkpoint_file = Path.cwd().parents[1] / "src" / "model_training" / "results" / "hyperparam_checkpoint_refine.json"
with open(checkpoint_file, "r") as f:
    checkpoint = json.load(f)
records = checkpoint["records"]
valid_records = [r for r in records if r["local_best_loss"] is not None]
top5 = sorted(valid_records, key=lambda x: x["local_best_loss"])[:5]

for i, r in enumerate(top5, 1):
    print("\n")
    print(f"Val loss : {r['local_best_loss']:.4f}")
    print(f"Epoch    : {r['local_best_epoch']}")
    for k, v in r["local_params"].items():
        print(f"{k}: {v}")



Val loss : 283.9819
Epoch    : 765
norm: tanh_norm
hidden_layers: [2048, 1024, 512]
learning_rate: 0.0001
input_dropout: 0.2
hidden_dropout: 0.3


Val loss : 284.9357
Epoch    : 654
norm: norm
hidden_layers: [512, 512]
learning_rate: 0.0001
input_dropout: 0.2
hidden_dropout: 0.3


Val loss : 285.0658
Epoch    : 300
norm: norm
hidden_layers: [2048, 1024, 512]
learning_rate: 0.0001
input_dropout: 0.2
hidden_dropout: 0.3


Val loss : 285.1789
Epoch    : 371
norm: tanh_norm
hidden_layers: [512, 512]
learning_rate: 0.0001
input_dropout: 0.2
hidden_dropout: 0.5


Val loss : 286.9894
Epoch    : 434
norm: tanh_norm
hidden_layers: [512, 512]
learning_rate: 0.0001
input_dropout: 0.2
hidden_dropout: 0.3
